In [0]:
dbutils.widgets.removeAll()

from pyspark.sql.functions import col, trim, upper, regexp_replace, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_smartdata_final")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdata2023")
dbutils.widgets.text("fileName", "Electric_Vehicle_Population_Data.csv")
dbutils.widgets.text("targetTable", "ev_population_bronze")
dbutils.widgets.dropdown("writeMode", "overwrite", ["overwrite", "append"])

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
file_name = dbutils.widgets.get("fileName")
target_table = dbutils.widgets.get("targetTable")
write_mode = dbutils.widgets.get("writeMode")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/{file_name}"
tabla_destino = f"{catalogo}.{esquema}.{target_table}"

print(f"Ruta origen: {ruta}")
print(f"Tabla destino: {tabla_destino}")

Ruta origen: abfss://raw@adlssmartdata2023.dfs.core.windows.net/Electric_Vehicle_Population_Data.csv
Tabla destino: catalog_smartdata_final.bronze.ev_population_bronze


In [0]:
ev_population_schema = StructType([
    StructField("VIN (1-10)", StringType(), True),
    StructField("County", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", StringType(), True),
    StructField("Model Year", IntegerType(), True),
    StructField("Make", StringType(), True),
    StructField("Model", StringType(), True),
    StructField("Electric Vehicle Type", StringType(), True),
    StructField("Clean Alternative Fuel Vehicle (CAFV) Eligibility", StringType(), True),
    StructField("Electric Range", IntegerType(), True),
    StructField("Base MSRP", IntegerType(), True),
    StructField("Legislative District", IntegerType(), True),
    StructField("DOL Vehicle ID", LongType(), True),
    StructField("Vehicle Location", StringType(), True),
    StructField("Electric Utility", StringType(), True),
    StructField("2020 Census Tract", LongType(), True)
])

In [0]:
df_population = spark.read\
    .option("header", True)\
    .schema(ev_population_schema)\
    .csv(ruta)

normalize_text = lambda c: upper(trim(regexp_replace(col(c), r"\\s+", " ")))

df_population_bronze = df_population.select(
    col("VIN (1-10)").alias("vin_1_10"),
    normalize_text("County").alias("county"),
    normalize_text("City").alias("city"),
    normalize_text("State").alias("state"),
    col("Postal Code").alias("postal_code"),
    col("Model Year").alias("model_year"),
    normalize_text("Make").alias("make"),
    normalize_text("Model").alias("model"),
    col("Electric Vehicle Type").alias("electric_vehicle_type"),
    col("Clean Alternative Fuel Vehicle (CAFV) Eligibility").alias("cafv_eligibility"),
    col("Electric Range").alias("electric_range"),
    col("Base MSRP").alias("base_msrp"),
    col("Legislative District").alias("legislative_district"),
    col("DOL Vehicle ID").alias("dol_vehicle_id"),
    col("Vehicle Location").alias("vehicle_location"),
    col("Electric Utility").alias("electric_utility"),
    col("2020 Census Tract").alias("census_tract_2020"),
    lit(file_name).alias("source_file"),
    current_timestamp().alias("ingestion_ts")
)

In [0]:
df_population_bronze.write.mode(write_mode).insertInto(tabla_destino)

print(f"Registros escritos: {df_population_bronze.count()}")
display(df_population_bronze.limit(10))

Registros escritos: 210165


vin_1_10,county,city,state,postal_code,model_year,make,model,electric_vehicle_type,cafv_eligibility,electric_range,base_msrp,legislative_district,dol_vehicle_id,vehicle_location,electric_utility,census_tract_2020,source_file,ingestion_ts
5UXTA6C0XM,KITSAP,SEABECK,WA,98380,2021,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,30,0,35,267929112,POINT (-122.8728334 47.5798304),PUGET SOUND ENERGY INC,53035091301,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
5YJ3E1EB1J,KITSAP,POULSBO,WA,98370,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,23,475911439,POINT (-122.6368884 47.7469547),PUGET SOUND ENERGY INC,53035091100,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
WP0AD2A73G,SNOHOMISH,BOTHELL,WA,98012,2016,PORSCHE,PANAMERA,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,15,0,1,101971278,POINT (-122.206146 47.839957),PUGET SOUND ENERGY INC,53061052009,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
5YJ3E1EB5J,KITSAP,BREMERTON,WA,98310,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,23,474363746,POINT (-122.6231895 47.5930874),PUGET SOUND ENERGY INC,53035080200,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
1N4AZ1CP3K,KING,REDMOND,WA,98052,2019,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,150,0,45,476346482,POINT (-122.13158 47.67858),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033032323,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
3FA6P0PU5F,SNOHOMISH,BOTHELL,WA,98012,2015,FORD,FUSION,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,19,0,21,9620047,POINT (-122.206146 47.839957),PUGET SOUND ENERGY INC,53061041704,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
5YJYGDEEXL,KING,RENTON,WA,98055,2020,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,291,0,11,124565731,POINT (-122.2003346 47.4487206),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033025805,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
5UXTS1C06M,KING,SEATTLE,WA,98107,2021,BMW,X3,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,17,0,36,135327104,POINT (-122.38591 47.67597),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),53033004702,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
1N4AZ0CP0F,KING,BELLEVUE,WA,98007,2015,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,84,0,48,105509778,POINT (-122.1436732 47.6157551),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033023300,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
5YJSA1E20H,KING,SEATTLE,WA,98125,2017,TESLA,MODEL S,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,210,0,46,348605603,POINT (-122.30253 47.72656),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),53033000900,Electric_Vehicle_Population_Data.csv,2026-03-15T23:47:33.441272Z
